# Chile climate-finance inventory — exploration

Supply-side profiling of the harmonized inventory (`chile_finance_inventory.csv`, built by `harmonize.py`).
Goal: understand what funding exists *before* matching it to actions — by sector, eligible actor, recurrence, specificity, and current usability. Methodology: `methodology.md`.

In [1]:
import pandas as pd, json
inv=pd.read_csv("data/chile_finance_inventory.csv")
print(inv.shape, "rows x cols")
inv.source_dataset.value_counts()

(78, 21) rows x cols


source_dataset
cl-mma           55
cl-minenergia     6
cl-corfo          5
cl-subdere        4
cl-minvu          4
cl-gore           4
Name: count, dtype: int64

## Usability flag (methodology §2a): open/rolling, or reliably annual/ongoing.

In [2]:
def usable(r):
    s=str(r["status"]).lower(); rec=str(r["recurrence"]).lower()
    if "open" in s or "rolling" in s: return True
    if rec.startswith("annual") or rec.startswith("ongoing"): return True
    return False
inv["usable_now"]=inv.apply(usable, axis=1)
inv.groupby(["source_dataset","usable_now"]).size().unstack(fill_value=0)

usable_now,False,True
source_dataset,,
cl-corfo,0,5
cl-gore,0,4
cl-minenergia,2,4
cl-minvu,0,4
cl-mma,27,28
cl-subdere,0,4


## Coverage by GPC sector (gpc_sectors is a JSON list — explode it).

In [3]:
ex=inv.assign(gpc=inv.gpc_sectors.apply(json.loads)).explode("gpc")
ex.groupby("gpc").agg(funds=("program_name","count"),
                      usable=("usable_now","sum")).sort_values("funds",ascending=False)

,funds,usable
gpc,,
cross_sector,47,30
afolu,16,6
waste,15,15
stationary_energy,13,11
industry,4,4
water,4,4
buildings,1,0


## Eligible actor — who can actually pursue these.

In [4]:
inv["actor_simple"]=inv.eligible_actor.str.extract(r'^(municipality|community/citizen org|household|indigenous community|research/university|school sostenedor|municipality \+ |municipality \()', expand=False).fillna(inv.eligible_actor.str.slice(0,24))
inv.eligible_actor.value_counts()

eligible_actor
unspecified                                                                            29
municipality                                                                           14
community/citizen org                                                                   7
indigenous community                                                                    7
research/university + ngo                                                               2
municipality + comités de pavimentación                                                 1
green-hydrogen project developers / firms                                               1
private firm / SME                                                                      1
private firm / growth-stage venture                                                     1
private firm (any size; no annual-sales cap)                                            1
legal persons / firms / institutions (productive investment; varies by region)       

## Recurrence & specificity — the scoring controls.

In [5]:
print("recurrence:\n", inv.recurrence.value_counts().to_string())
print("\nspecificity (broad funds are capped at 'Moderate' in scoring):")
print(inv.groupby("specificity").program_name.count().to_string())
print("\nbroad funds:", inv[inv.specificity=='broad'].program_name.tolist())

recurrence:
 recurrence
annual                                                            30
sporadic                                                          16
one-off                                                           11
ongoing (programme)                                                3
ongoing/periodic                                                   3
ongoing                                                            2
ongoing (rolling, budget-dependent)                                1
ongoing (annual, per GORE)                                         1
ongoing (annual budget)                                            1
ongoing (construction prioritised) + conservation programme        1
annual (Regular call ~April; Especial ~September)                  1
sporadic/periodic                                                  1
ongoing (rolling, all year)                                        1
sporadic (created 2014 as pilot; last call closed 20 Jul 2024)     1
ongoing (r

## Where could 'Strong' matches come from? sector-specific + usable_now + explicit climate relevance.

In [6]:
strong_supply=inv[(inv.specificity=="sector-specific") & (inv.usable_now) & (inv.climate_relevance_norm=="explicit")]
print(len(strong_supply),"funds qualify as Strong-eligible supply")
strong_supply[["source_dataset","program_name","eligible_actor","recurrence"]].head(20)

39 funds qualify as Strong-eligible supply


,source_dataset,program_name,eligible_actor,recurrence
0,cl-mma,FPA 2026 - Proyectos Sustentables Ciudadanos,community/citizen org,annual
1,cl-mma,FPA 2026 - Proyectos Sustentables en Estableci...,community/citizen org,annual
2,cl-mma,FPA 2026 - Proyectos Sustentables para Pueblos...,indigenous community,annual
6,cl-mma,FPR 2026 - Fondo para el Reciclaje,municipality,annual
16,cl-mma,FPA 2024 – Fortalecimiento para recicladores d...,unspecified,annual
18,cl-mma,FPA 2024 – Chiloé Reduce en tu Establecimiento...,unspecified,annual
21,cl-mma,FPA 2023 – Proyectos Sustentables para Pueblos...,indigenous community,annual
22,cl-mma,FPA 2023 – Proyectos Sustentables en Estableci...,unspecified,annual
23,cl-mma,FPA 2023 – Proyectos Sustentables Ciudadanos,unspecified,annual
24,cl-mma,FPA 2022 – Emprendimientos Verdes para Comunid...,indigenous community,annual


## Read-out (exploratory)

- The inventory is **mitigation-heavy and currently mostly closed** (annual environment funds between cycles), so usability hinges on the `recurrence=annual` rule, not live `status`.
- **Broad funds are few but powerful** (PMU/PMB/FRC/PMU-type) — capped at Moderate by design so they don't inflate.
- Sector coverage is strongest in **waste / cross_sector / stationary_energy**, with urban-green (afolu) from MINVU and adaptation thin (mainly SUBDERE risk).
- Next: bring the **action set** in and run the match → tier per `methodology.md`; tune cutoffs here.